# DRfold original · Kaggle GPU T4 · Pilot10 remaining 8 targets

**Purpose:** Continue the same DRfold checkpoint used in the verified Colab experiment. Completed original-pilot targets `9g4p_B` and `8ubd_H` have already been archived in Google Drive and **must not be rerun**. `9fia_bJ` failed when Colab resources ran out and is scheduled **last**. Eight other targets, sorted shortest-first, are pending.

**Not DRfold2. No native experimental PDB enters inference.** Sequence inputs are exactly the eight frozen FASTAs (SHA-256 checked), retrieved from the previously archived trRosettaRNA2 inputs that came from the frozen NuFold dataset. Input mode = sequence plus *internally derived* PETfold/ViennaRNA features, **not** a genuine MSA experiment.

## Before running
1. Kaggle → **Settings → Accelerator → GPU T4 ×2**; **Turn on internet**. GPU 0 is used; dual-GPU acceleration is **not** claimed.
2. Run cells from top to bottom. The setup uses the **pinned, verified** environment/bootstrap recipe from the benchmark repository (Python 3.9, old PyTorch, ViennaRNA, PETfold, OpenMM) in a separate Micromamba environment. It downloads model weights and can fail if Kaggle's image/network restrictions change.
3. **Default: one new target per notebook run**, first `8z8u_B` (41 nt). Set `TARGET_TO_RUN` in section 1 to another **pending** target when ready. No GPU activity until section 3.
4. **Crucial Kaggle persistence limitation:** `/kaggle/working` is NOT an immediately permanent Google Drive mount. After each completed prediction, download the generated `drfold_checkpoint_TARGET.zip` **or save a Kaggle notebook version that retains outputs**. On the next session, attach that ZIP as Kaggle Dataset Input; section 1 automatically imports all matching checkpoint ZIPs. If a notebook crashes before a target completes, that target is not recoverable from this notebook.
5. Do not run all eight blindly. Long RNA (>60 nt) could take 1–2+ hours. The notebook records time but cannot guarantee completion within Kaggle quotas.

**Data policy:** no GitHub writes; no overwrite of verified outputs; no cloud Drive write. Raw PDB/logs are stored in a per-target ZIP to download and later transfer to the existing `RNA3D_Benchmark/DRfold/runs/drfold_pilot10_single_20261115_v1/` folder without changing earlier targets.

## 1 · Hardware, frozen RNA sequences, and previous checkpoint imports

In [ ]:
from pathlib import Path
import os, sys, json, csv, hashlib, re, time, shutil, zipfile, threading, subprocess, traceback
from datetime import datetime, timezone

assert Path('/kaggle/working').is_dir(), 'Run this notebook on Kaggle.'
print(subprocess.run(['nvidia-smi','-L'],capture_output=True,text=True,check=True).stdout)
assert subprocess.check_output(['nvidia-smi','-L'],text=True).count('Tesla T4')>=1, 'Expected at least one T4 GPU'

UPSTREAM_SHA = '9586990c79e4ca488e5f967fcc8bd5b06cd60273'
PROJECT_SHA = '2946e14411b75be1621635164733d25c4d4113de'
FROZEN_FASTAS = {'8z8u_B': 'GGGCAGAGCCCAACACAGCGAAAGCUGUGGCUAGACUGUCC', '9g6k_lB': 'UGUGUGAGAAGGAGCUAUACAGAUACAGGCGAAGGAGAAGCGGGA', '9lq7_K': 'CACAAGGGACGCCACUUCAUGGAAAUAAAUCACUCACGAAUGUCUUAAU', '9m31_B': 'AACCCCUACCAACUGGUCGGGGUUUGAAACGCCAACGAACAUCAUGACAGC', '9cfn_B': 'AAGUACCCUCCAAGCCCUACAGGUUGGAAGAGGGGGCUAUCAGUCCUGUAGGCAGACUC', '9cf2_W': 'UUAUCCACCAAAGUUAUCGCUUUGGUCAAUUAAUGCAGGUAAGCAACAUCCAGCAAACAGA', '8wf8_B': 'UCUAAACCAUCCUGCGGCCUCUACUCUGCAGUUGUGGAAGGUCCAGUUUUGAGGGGCUAUUACAAC', '9fia_bJ': 'CAUAGAAGAUUAAGCUGCAUUACGUUUAAACUUUACACUGGAUACGUUUUAAUGUUAACUUACUACAGACC'}
EXPECTED_FASTA_SHA256 = {'8z8u_B': 'e63938c2c7ece1367b8c9ec724dfe09b35d871fcc709588b579eca779cd1f175', '9g6k_lB': '418c6e1663cb576bab1f245e04be6a69ff981fac798594cca185e3863f10f5a8', '9lq7_K': '37703a204aab609686b3c967f93ca480186afa6f6418b6bec79aa7da0c2e68c8', '9m31_B': '4cd32e93ac002a73fc809110130dda0f170ecad48ffaa792792503a4a79d2288', '9cfn_B': '6cc443254acbb6506ef46d78f48ac23e1215ec72a63ad60bcdb2dc846a3ca3c9', '9cf2_W': '80a35e48c4bbebbfde00681f212988808d3269fcf84cfd10584e8f366d7c0a7f', '8wf8_B': 'c97700fd2c42ef92535847c279a804c246c3d5dbc033aae1036cd119ec64bd85', '9fia_bJ': '48db0b404d1c1d87630d0f56f016b2dadcfa1a7b2c9fb3fc8ef2c00f5846ef3e'}
ALREADY_COMPLETED_ON_DRIVE = ['9g4p_B','8ubd_H']
RUN_ID = 'drfold_pilot10_single_20261115_v1'
TARGET_TO_RUN = '8z8u_B'  # Change only to another pending target, one per run.
SETUP_ROOT=Path('/kaggle/temp/drfold_setup')
RESULTS=Path('/kaggle/working/drfold_results')
RESULTS.mkdir(parents=True,exist_ok=True)

def sha256(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for part in iter(lambda:f.read(1024*1024),b''):h.update(part)
    return h.hexdigest()

def verify_result(t):
    d=RESULTS/t
    if not d.exists():return False
    required=[d/(t+'.fasta'),d/'DPR.pdb',d/'inference.log',d/'complete.json']
    if not all(p.is_file() for p in required):raise RuntimeError('Partial checkpoint: '+str(d))
    meta=json.loads((d/'complete.json').read_text())
    assert meta['target_id']==t and meta['model_commit']==UPSTREAM_SHA and meta['status']=='PASS'
    assert meta['native_used_for_inference'] is False
    assert meta['input_length']==len(FROZEN_FASTAS[t]) and meta['output_residues']==len(FROZEN_FASTAS[t])
    assert sha256(d/(t+'.fasta'))==EXPECTED_FASTA_SHA256[t]==meta['input_fasta_sha256']
    assert sha256(d/'DPR.pdb')==meta['prediction_sha256']
    return True

# Import checkpoint ZIPs attached as read-only Kaggle Dataset inputs.
# Each file must have been generated by this notebook (drfold_results/TARGET/...).
archives=sorted(Path('/kaggle/input').rglob('drfold_checkpoint_*.zip'))
for archive in archives:
    with zipfile.ZipFile(archive) as z:
        names=z.namelist()
        assert names and all(not n.startswith('/') and '..' not in Path(n).parts for n in names)
        target=archive.stem.replace('drfold_checkpoint_','')
        if target not in FROZEN_FASTAS:continue
        expected_prefix=f'drfold_results/{target}/'
        assert all(n.startswith(expected_prefix) for n in names), f'Unexpected ZIP contents: {archive}'
        dest=RESULTS/target
        if dest.exists():
            assert verify_result(target),f'Existing checkpoint invalid: {target}'
            print('Already imported:',target);continue
        z.extractall('/kaggle/working')
        assert verify_result(target),f'Imported checkpoint failed verification: {archive}'
        print('Imported verified checkpoint:',target)

for target,seq in FROZEN_FASTAS.items():
    assert re.fullmatch(r'[ACGU]+',seq) and len(seq)>0
    raw=f'>{target}\n{seq}\n'.encode()
    assert hashlib.sha256(raw).hexdigest()==EXPECTED_FASTA_SHA256[target]
    print(target,len(seq),'DONE' if verify_result(target) else 'PENDING')
assert TARGET_TO_RUN in FROZEN_FASTAS
assert not verify_result(TARGET_TO_RUN),f'Already done: {TARGET_TO_RUN}. Choose a pending one.'
assert not (RESULTS/TARGET_TO_RUN).exists(),f'Partial target folder present, inspect: {TARGET_TO_RUN}'
print('NEXT TARGET:',TARGET_TO_RUN, len(FROZEN_FASTAS[TARGET_TO_RUN]), 'nt')


## 2 · Install pinned DRfold dependencies (GPU setup; run once per session)

This executes only the **setup and feature preflight** code cells of the previously verified Colab notebook. All paths are translated from `/content` to a private writable Kaggle temporary folder. It intentionally **does not execute** the bundled 77-nt DRfold example or the Colab-only `drive.mount` cells. If a package fails, stop; do not silently substitute a different model or checkpoint.

**First-time setup may download large files and take significant time.** Kaggle internet must be ON. The compatibility of this legacy Colab recipe with Kaggle's current image is not independently verified; the cell will stop visibly if it fails.

In [ ]:
SETUP_ROOT.mkdir(parents=True,exist_ok=True)
os.environ['CUDA_VISIBLE_DEVICES']='0'   # DRfold original is a one-GPU pipeline
os.environ.setdefault('OMP_NUM_THREADS','4')
project=SETUP_ROOT/'rna3d-drfold-setup'
repo=SETUP_ROOT/'DRfold'

def cmd(args,cwd=None,env=None):
    args=[str(x) for x in args]
    print('$',' '.join(args),flush=True)
    subprocess.run(args,check=True,cwd=cwd,env=env)

if not project.is_dir():cmd(['git','clone','--filter=blob:none','https://github.com/ABDULSALAM589/rna3d-template-sparse-benchmark.git',project])
cmd(['git','-C',project,'checkout','--detach',PROJECT_SHA])
assert subprocess.check_output(['git','-C',project,'rev-parse','HEAD'],text=True).strip()==PROJECT_SHA
if not repo.is_dir():cmd(['git','clone','https://github.com/leeyang/DRfold.git',repo])
cmd(['git','-C',repo,'checkout','--detach',UPSTREAM_SHA])
assert subprocess.check_output(['git','-C',repo,'rev-parse','HEAD'],text=True).strip()==UPSTREAM_SHA
try:
    import nbformat
except ImportError:
    cmd([sys.executable,'-m','pip','install','nbformat'])
    import nbformat
source=nbformat.read(project/'notebooks/drfold_colab.ipynb',as_version=4)
steps=[(i,c) for i,c in enumerate(source.cells[:29]) if c.cell_type=='code']
assert [i for i,c in steps]==list(range(2,29,2)), 'Pinned source notebook changed unexpectedly'

# Original setup notebook uses /content paths, %cd and !shell commands.
# IPython executes these natively on Kaggle; path replacement is restricted to the setup code.
for i,c in steps:
    adapted=c.source.replace('/content',str(SETUP_ROOT))
    if i==4:
        # Git HEAD has been checkout-pinned before calling the source setup cell.
        pass
    print(f'=== DRfold pinned environment setup: original cell {i}/28 ===',flush=True)
    result=get_ipython().run_cell(adapted)
    if result.error_before_exec or result.error_in_exec:
        raise RuntimeError(f'DRfold setup failed in original cell {i}; inspect displayed traceback') from (result.error_in_exec or result.error_before_exec)

PYTHON_EXE=SETUP_ROOT/'micromamba_root/envs/drfold/bin/python'
DRFOLD_SH=repo/'DRfold.sh'
VIENNA_PREFIX=SETUP_ROOT/'micromamba_root/envs/drfold'
PETFOLD_REAL=globals().get('PETFOLD_REAL')
assert PYTHON_EXE.is_file() and DRFOLD_SH.is_file()
assert PETFOLD_REAL is not None and Path(PETFOLD_REAL).is_file()
assert (VIENNA_PREFIX/'bin/RNAfold').is_file()
assert subprocess.check_output(['git','-C',repo,'rev-parse','HEAD'],text=True).strip()==UPSTREAM_SHA
run_env=os.environ.copy()
run_env['PATH']=f"{VIENNA_PREFIX/'bin'}:{Path(PETFOLD_REAL).parent}:"+run_env['PATH']
run_env['LD_LIBRARY_PATH']=f"{VIENNA_PREFIX/'lib'}:"+run_env.get('LD_LIBRARY_PATH','')
run_env['CUDA_VISIBLE_DEVICES']='0'
print('ENVIRONMENT PREFLIGHT PASS — DRfold pinned setup ready')


## 3 · Run exactly one pending RNA and save a checked checkpoint ZIP

The subprocess runs **uninterrupted**, prints progress every five minutes, captures stdout/stderr to `inference.log`, and saves the verified `DPR.pdb` plus metadata to `/kaggle/working`. It does **not** overwrite any target. **Wait for `CHECKPOINT SAVED` before stopping the session.** Target `9fia_bJ` (71 nt) may exceed available quota; run it last.

In [ ]:
assert TARGET_TO_RUN in FROZEN_FASTAS and not verify_result(TARGET_TO_RUN)
STAGE=SETUP_ROOT/'pilot10_stage'/TARGET_TO_RUN
if STAGE.exists():raise FileExistsError(f'Local staging exists: {STAGE}. Inspect before running.')
STAGE.mkdir(parents=True)
SEQ=FROZEN_FASTAS[TARGET_TO_RUN]
FASTA=STAGE/(TARGET_TO_RUN+'.fasta')
FASTA.write_text('>'+TARGET_TO_RUN+'\n'+SEQ+'\n')
assert sha256(FASTA)==EXPECTED_FASTA_SHA256[TARGET_TO_RUN]
OUTPUT=STAGE/'model_output'
LOG=STAGE/'inference.log'
readings=[]; stop=threading.Event()
def gpu_monitor():
    while not stop.wait(10):
        try:
            raw=subprocess.check_output(['nvidia-smi','-i','0','--query-gpu=memory.used','--format=csv,noheader,nounits'],text=True,timeout=10)
            readings.append(int(raw.strip().splitlines()[0]))
        except Exception:pass
watcher=threading.Thread(target=gpu_monitor,daemon=True);watcher.start()
print('START:',TARGET_TO_RUN,'length:',len(SEQ),'GPU: T4 #0',flush=True)
started=time.monotonic();returncode=None
try:
    with LOG.open('w') as logfile:
        p=subprocess.Popen(['bash',str(DRFOLD_SH),str(FASTA),str(OUTPUT)],cwd=repo,env=run_env,stdout=logfile,stderr=subprocess.STDOUT)
        while p.poll() is None:
            try:p.wait(timeout=300)
            except subprocess.TimeoutExpired:
                print(f'{TARGET_TO_RUN}: elapsed {(time.monotonic()-started)/60:.1f} min, max observed VRAM {max(readings) if readings else None} MiB',flush=True)
        returncode=p.returncode
finally:
    stop.set();watcher.join(timeout=3)
elapsed=round(time.monotonic()-started,3)
PDB=OUTPUT/'DPR.pdb'
if returncode!=0 or not PDB.is_file() or PDB.stat().st_size<1000:
    print(LOG.read_text(errors='replace')[-5000:]);raise RuntimeError(f'DRfold failed exit={returncode} on {TARGET_TO_RUN}, not saved')
residue_keys=set()
for line in PDB.read_text(errors='replace').splitlines():
    if line.startswith('ATOM  '):residue_keys.add((line[21],line[22:26],line[26]))
assert len(residue_keys)==len(SEQ),f'PDB residues={len(residue_keys)} vs expected={len(SEQ)}'

meta={'target_id':TARGET_TO_RUN,'model':'DRfold','model_commit':UPSTREAM_SHA,
      'status':'PASS','input_mode':'sequence_with_internal_petfold_viennarna',
      'input_length':len(SEQ),'output_residues':len(residue_keys),
      'runtime_seconds':elapsed,'peak_gpu_memory_mib_observed':max(readings) if readings else None,
      'input_fasta_sha256':sha256(FASTA),'prediction_sha256':sha256(PDB),'exit_code':returncode,
      'native_used_for_inference':False,'execution_platform':'Kaggle T4',
      'created_utc':datetime.now(timezone.utc).isoformat()}
DEST=RESULTS/TARGET_TO_RUN
assert not DEST.exists(),f'Refusing to overwrite completed target: {DEST}'
DEST.mkdir(parents=True,exist_ok=False)
shutil.copy2(FASTA,DEST/FASTA.name)
shutil.copy2(PDB,DEST/'DPR.pdb')
shutil.copy2(LOG,DEST/'inference.log')
assert sha256(DEST/FASTA.name)==meta['input_fasta_sha256']
assert sha256(DEST/'DPR.pdb')==meta['prediction_sha256']
(DEST/'complete.json').write_text(json.dumps(meta,indent=2)+'\n')
assert verify_result(TARGET_TO_RUN)
ARCHIVE=Path('/kaggle/working')/f'drfold_checkpoint_{TARGET_TO_RUN}.zip'
assert not ARCHIVE.exists()
with zipfile.ZipFile(ARCHIVE,'x',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for f in sorted(DEST.iterdir()):z.write(f,arcname=f'drfold_results/{TARGET_TO_RUN}/{f.name}')
print('CHECKPOINT SAVED:',TARGET_TO_RUN)
print('DOWNLOAD / SAVE VERSION NOW:',ARCHIVE)
print('Runtime minutes:',round(elapsed/60,2),'PDB SHA256:',meta['prediction_sha256'])


## 4 · Read-only checkpoint inventory and next target

**Before closing Kaggle:** download your `drfold_checkpoint_TARGET.zip` from the right-side **Output** panel or use **Save Version**; then attach previous ZIPs as **Input** on your next session. Uploading them to a Kaggle Dataset is one convenient way to attach them. **Do not lose ZIPs**—Kaggle working files are not equivalent to immediately synced Drive files.

Successful Kaggle outputs can later be copied to existing Drive run under each target's folder (`DPR.pdb`, `<target>.fasta`, `inference.log`, `complete.json`) without overwriting archived Colab targets.

In [ ]:
print('Already archived on Drive (not downloaded here):',ALREADY_COMPLETED_ON_DRIVE)
for t in FROZEN_FASTAS:
    print(f'{t:10s}  {"PASS" if verify_result(t) else "PENDING":8s}',f'{len(FROZEN_FASTAS[t])} nt')
print('Kaggle checkpoint archives available to download:')
for p in sorted(Path('/kaggle/working').glob('drfold_checkpoint_*.zip')):
    print(' ',p,p.stat().st_size,'bytes')
print('NOTE: Kaggle results do not automatically sync to Drive. Do not claim 10/10 until all target outputs are collected and verified.')
